# 02 — Comparaison de prompts (baseline vs amélioré)

Exécuter les deux prompts sur les mêmes cas et comparer les métriques
(accuracy, macro-F1, sensibilité, spécificité, JSON valide, latence).

Sur GPU, remplacer `backend='toy'` par `backend='vlm'` et `--cases data/rsna/cases.csv --split final`.

In [ ]:
from pathlib import Path
import sys, csv
sys.path.append(str(Path('..').resolve()))
from src.inference import predict
from src.guardrails import apply_safety_guardrails, validate_prediction
from src.metrics import summarize_metrics, classify_error

BACKEND = 'toy'
with open('../data/synthetic_cases.csv', newline='', encoding='utf-8') as f:
    cases = list(csv.DictReader(f))

def evaluate(mode):
    rows = []
    for c in cases:
        pred = apply_safety_guardrails(predict(Path('..') / c['image_path'], mode=mode, backend=BACKEND))
        valid, _ = validate_prediction(pred)
        rows.append({'label': c['label'], 'predicted_class': pred['predicted_class'],
                     'json_valid': valid, 'warning': pred['warning'],
                     'latency_ms': pred.get('latency_ms', 0),
                     'error_type': classify_error(c['label'], pred['predicted_class'], valid)})
    return summarize_metrics(rows)

{'baseline': evaluate('baseline'), 'improved': evaluate('improved')}

Équivalent en ligne de commande (génère `before_after_summary.csv`) :

```bash
python ../eval/run_evaluation.py --mode toy --out-dir /tmp/eval --db-path /tmp/runs.sqlite
```